# xView2 Damage Model — Grad-CAM Only (pulls everything from GitHub + Kaggle)

This notebook does **not** need anything from your Downloads folder. Each time you run it, it:

1. Clones (or updates) `aishanikar9/BWSI_Operations_Team` from GitHub — this is where `resnet50-damage_model_BEST.pth` lives.
2. Downloads the Kaggle xView2 dataset via `kagglehub` (you'll need a free Kaggle account/API key the first time — Kaggle will prompt you).
3. Pulls out a handful of building chips (one per damage class) straight from the raw Kaggle images — it stops scanning as soon as it has what it needs, instead of re-cropping the whole ~30k-chip dataset.
4. Loads the trained ResNet-50 checkpoint and runs Grad-CAM on those chips.
5. Saves the overlay JPGs into a **`gradcam_outputs/` folder inside the cloned repo** (not your Downloads folder). The folder is wiped and re-written on every run, so old images never pile up.

Run the cells top to bottom. If you're on Colab, turn on a GPU runtime (Runtime → Change runtime type → GPU) — it's optional but faster.

In [ ]:
%pip install -q torch torchvision opencv-python-headless shapely kagglehub tqdm pillow pandas numpy matplotlib

## 1. Clone the GitHub repo

Public repo, so no token needed just to read it. If the folder already exists (e.g. you're re-running this), it just `git pull`s the latest instead of re-cloning.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/aishanikar9/BWSI_Operations_Team.git"
REPO_DIR = Path.cwd() / "BWSI_Operations_Team"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

CHECKPOINT_PATH = REPO_DIR / "resnet50-damage_model_BEST.pth"
assert CHECKPOINT_PATH.exists(), f"Checkpoint not found at {CHECKPOINT_PATH}"
print("Repo ready at:", REPO_DIR)
print("Checkpoint found:", CHECKPOINT_PATH)

## 2. Download the Kaggle dataset

The first time this runs, `kagglehub` may ask you to authenticate (either a browser popup or a `kaggle.json` API key — see https://www.kaggle.com/docs/api). After that it caches the dataset locally so re-runs are fast.

In [ ]:
import kagglehub

kaggle_path = kagglehub.dataset_download("tunguz/xview2-challenge-dataset-train-and-test")
print("Kaggle dataset at:", kaggle_path)

## 3. Pull a few sample building chips (one per damage class)

Same cropping logic as the original notebook (bounding box from the polygon `wkt`), but it **stops as soon as it has one `no-damage`, one `minor-damage`, one `major-damage`, one `destroyed` chip** (plus a first "reference" chip) — so it doesn't need to process the whole dataset just to make a few Grad-CAM images.

In [ ]:
import os
import json
import cv2
from shapely.wkt import loads as wkt_loads

DAMAGE_MAP = {"no-damage": 0, "minor-damage": 1, "major-damage": 2, "destroyed": 3}
CLASS_NAMES = ["no-damage", "minor-damage", "major-damage", "destroyed"]

SAMPLES_DIR = REPO_DIR / "gradcam_outputs" / "sample_chips"
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

train_dir = os.path.join(kaggle_path, "train", "train")
train_image_dir = os.path.join(train_dir, "images")
train_label_dir = os.path.join(train_dir, "labels")

post_jsons = sorted(f for f in os.listdir(train_label_dir) if f.endswith("_post_disaster.json"))

samples = {}          # damage_label -> chip path
first_chip = None     # (chip_path, damage_label) for a single "reference" example
crop_counter = 0

for j_file in post_jsons:
    if first_chip is not None and len(samples) == 4:
        break

    img_file = j_file.replace(".json", ".png")
    img_path = os.path.join(train_image_dir, img_file)
    if not os.path.exists(img_path):
        continue

    img = cv2.imread(img_path)
    if img is None:
        continue

    with open(os.path.join(train_label_dir, j_file)) as f:
        label_data = json.load(f)

    for feature in label_data["features"]["xy"]:
        damage_type = feature["properties"]["subtype"]
        if damage_type not in DAMAGE_MAP:
            continue
        damage_label = DAMAGE_MAP[damage_type]

        # skip if we already have this class (unless we still need the "first" reference chip)
        if damage_label in samples and first_chip is not None:
            continue

        try:
            poly = wkt_loads(feature["wkt"])
            xmin, ymin, xmax, ymax = poly.bounds
            xmin, ymin = max(0, int(xmin)), max(0, int(ymin))
            xmax, ymax = min(img.shape[1], int(xmax)), min(img.shape[0], int(ymax))
            crop = img[ymin:ymax, xmin:xmax]
            if crop.size == 0 or crop.shape[0] < 10 or crop.shape[1] < 10:
                continue
        except Exception:
            continue

        crop_counter += 1
        chip_path = SAMPLES_DIR / f"sample_building_{crop_counter}.png"
        cv2.imwrite(str(chip_path), crop)

        if first_chip is None:
            first_chip = (chip_path, damage_label)
        if damage_label not in samples:
            samples[damage_label] = chip_path

    if first_chip is not None and len(samples) == 4:
        break

print("Reference chip:", first_chip)
print("Per-class chips:")
for label, path in sorted(samples.items()):
    print(f"  {CLASS_NAMES[label]}: {path}")

## 4. Load the model + checkpoint

Same ResNet-50 + dropout/linear head architecture the model was trained with.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

net = torchvision.models.resnet50(weights=None)
net.fc = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(2048, 4)
)
net = net.to(device)
net.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
net.eval()

composed_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("Model loaded and ready.")

## 5. Grad-CAM function — fixed for sharper, bigger output

Two changes from the original cell fix the blurry/tiny images:

- **Target layer:** the original code hooked `layer1[-1]`. That's the *first* residual block — it sees edges/textures, not "this looks like storm damage," so the heatmap it produces is noisy and not very meaningful. This version hooks `layer4[-1]`, the last conv block, which is what standard Grad-CAM implementations use because it's the most class-discriminative layer.
- **Resizing:** the original code upsampled the heatmap with `INTER_NEAREST` (blocky/pixelated) and saved the overlay at the *original* building-chip resolution — and building chips are often only a few dozen pixels across, so the saved JPG was tiny. This version upscales the chip and the heatmap together (to at least 600px on the long side) using smooth interpolation (`INTER_LANCZOS4` for the image, `INTER_CUBIC` for the heatmap), so the saved file is both bigger and smoother.

One honest caveat: upscaling a tiny crop makes it *look* bigger and smoother, but it can't add real detail that wasn't in the original pixels — so extremely small building chips will still look soft close up. It just won't look blocky/pixelated anymore.

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

OUTPUT_DIR = REPO_DIR / "gradcam_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Wipe previously saved overlays so every run leaves a clean, current set (no pile-up)
for old_file in OUTPUT_DIR.glob("gradcam_*.jpg"):
    old_file.unlink()

MIN_OUTPUT_SIDE = 600  # saved overlays are upscaled to at least this many pixels on the long side


def run_gradcam(chip_path, true_label=None, save_path=None):
    forward_data = {"activation": None}

    def forward_hook(_module, _inputs, output):
        forward_data["activation"] = output

    # layer4 = last conv block = most class-discriminative Grad-CAM target
    target_layer = net.layer4[-1]
    forward_handle = target_layer.register_forward_hook(forward_hook)

    try:
        raw_image = Image.open(chip_path).convert("RGB")
        img_tensor = composed_val(raw_image).unsqueeze(0).to(device)
        img_tensor.requires_grad = True

        net.zero_grad(set_to_none=True)
        logits = net(img_tensor)
        predicted_class = logits.argmax(dim=1).item()

        feature_map = forward_data["activation"]
        score = logits[0, predicted_class]

        gradients = torch.autograd.grad(outputs=score, inputs=feature_map, retain_graph=True)
        feature_map_np = feature_map.detach().cpu().squeeze(0)
        gradient_np = gradients[0].detach().cpu().squeeze(0)

        weights = gradient_np.mean(dim=(1, 2), keepdim=True)
        heatmap = (weights * feature_map_np).sum(dim=0)
        heatmap = torch.relu(heatmap)
        heatmap = heatmap / (heatmap.max() + 1e-8)
        heatmap = heatmap.numpy()
    finally:
        forward_handle.remove()

    label_str = f" | True: {CLASS_NAMES[true_label]}" if true_label is not None else ""
    print(f"Predicted: {CLASS_NAMES[predicted_class]}{label_str}")

    img_cv = cv2.imread(str(chip_path))
    h, w = img_cv.shape[:2]
    scale = max(1, round(MIN_OUTPUT_SIDE / max(h, w)))
    out_w, out_h = w * scale, h * scale

    # Smooth upscale of the base chip (LANCZOS4 keeps enlarged edges crisp)
    img_big = cv2.resize(img_cv, (out_w, out_h), interpolation=cv2.INTER_LANCZOS4)

    # Smooth heatmap upsampling (CUBIC instead of NEAREST removes the blocky look)
    heatmap_resized = cv2.resize(heatmap, (out_w, out_h), interpolation=cv2.INTER_CUBIC)
    heatmap_resized = np.clip(heatmap_resized, 0, 1)
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)

    overlay = np.clip(heatmap_color * 0.4 + img_big, 0, 255).astype("uint8")

    # High-res inline preview
    plt.figure(figsize=(6, 5), dpi=150)
    plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    plt.title(f"Grad-CAM: predicted {CLASS_NAMES[predicted_class]}")
    plt.axis("off")
    plt.show()

    if save_path is None:
        save_path = OUTPUT_DIR / "gradcam_map.jpg"
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(save_path), overlay, [cv2.IMWRITE_JPEG_QUALITY, 95])
    print(f"Saved overlay to {save_path}")
    return predicted_class

## 6. Run it

Saves into `BWSI_Operations_Team/gradcam_outputs/` as `gradcam_row0.jpg` and `gradcam_class0.jpg` … `gradcam_class3.jpg`, overwriting whatever was there from a previous run.

In [ ]:
first_path, first_label = first_chip
run_gradcam(first_path, true_label=first_label, save_path=OUTPUT_DIR / "gradcam_row0.jpg")

for damage_class, chip_path in sorted(samples.items()):
    print(f"\n--- Damage class: {CLASS_NAMES[damage_class]} ---")
    run_gradcam(
        chip_path,
        true_label=damage_class,
        save_path=OUTPUT_DIR / f"gradcam_class{damage_class}.jpg",
    )

print("\nAll Grad-CAM overlays saved to:", OUTPUT_DIR)

## 7. (Optional) Push the new images back to GitHub

Saving to `gradcam_outputs/` above only writes to your **local clone** of the repo — it doesn't touch GitHub itself until you commit and push. If you want this notebook to publish the images back to GitHub automatically, generate a [Personal Access Token](https://github.com/settings/tokens) with `repo` scope, paste it below, and run the cell. Otherwise, just commit/push manually from a terminal (`cd BWSI_Operations_Team && git add gradcam_outputs && git commit -m "Update Grad-CAM outputs" && git push`).

**Heads up:** the cell below embeds your token in the git remote URL for this one push — don't share the notebook after running it with a real token pasted in, and consider revoking the token afterward if this is a shared machine.

In [ ]:
# GITHUB_TOKEN = "paste_your_personal_access_token_here"
#
# import subprocess
# subprocess.run(["git", "add", "gradcam_outputs"], cwd=REPO_DIR, check=True)
# subprocess.run(["git", "commit", "-m", "Update Grad-CAM outputs"], cwd=REPO_DIR, check=True)
# subprocess.run(
#     ["git", "push", f"https://{GITHUB_TOKEN}@github.com/aishanikar9/BWSI_Operations_Team.git"],
#     cwd=REPO_DIR,
#     check=True,
# )